# AgentCore Policy Lab
## Mastering Amazon Bedrock AgentCore | Pumping Code

---

## 🎯 What You'll Build

In this lab you will build a **deterministic authorization layer** around an Insurance Underwriting AI Agent using **Amazon Bedrock AgentCore Policy** and **Cedar policies**.

By the end of this lab, you will have:
- ✅ Deployed 3 Lambda functions as tool backends
- ✅ Created an AgentCore Gateway with OAuth authentication
- ✅ Run an agent with unrestricted tool access
- ✅ Created and attached a Policy Engine
- ✅ Observed **Default Deny** (empty policy engine blocks everything)
- ✅ Written Cedar policies and tested **ALLOW** and **DENY** scenarios
- ✅ Used **NL2Cedar** to generate policies from natural language

---

## 🏗️ Architecture

```
┌──────────────────┐
│    AI Agent      │  ← Local Strands Agent (Claude Haiku 4.5) 
└────────┬─────────┘
         │  MCP Tool Call
         ▼
┌──────────────────┐
│ AgentCore Gateway│  ← OAuth2 Auth (Cognito)
│ + Policy Engine  │  ← Cedar Policy Enforcement
└────────┬─────────┘
         │  ALLOW or DENY
         ▼
┌──────────────────┐
│  Lambda Targets  │  ← Application | Risk Model | Approval tools
└──────────────────┘
```

---

## ✅ Prerequisites
- AWS CLI configured with appropriate credentials
- Python 3.10+
- Access to AWS Lambda, Cognito, and Bedrock
- Bedrock access enabled for: `us.anthropic.claude-haiku-4-5-20251001-v1:0`

---
# Part 1: Environment Setup

In [ ]:
# Install required packages using uv when available
# Run this in your terminal if you prefer:
#   uv pip install --python $(which python) bedrock-agentcore-starter-toolkit boto3 strands-agents strands-agents-tools requests

import shutil
import subprocess
import sys

packages = [
    "bedrock-agentcore-starter-toolkit",
    "boto3",
    "strands-agents",
    "strands-agents-tools",
    "requests",
]

if shutil.which("uv"):
    subprocess.check_call(["uv", "pip", "install", "--python", sys.executable, *packages], stdout=subprocess.DEVNULL)
    print("✅ All packages installed via uv")
else:
    for pkg in packages:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", pkg, "-q"])

    print("✅ All packages installed via pip fallback")

In [ ]:
import os

os.environ['AWS_REGION'] = 'us-east-1'

# APPROACH A: Use credentials
# os.environ['AWS_ACCESS_KEY_ID'] = 'your_access_key'
# os.environ['AWS_SECRET_ACCESS_KEY'] = 'your_secret_key'
# os.environ['AWS_SESSION_TOKEN'] = "your_session_token"

# APPROACH B: Use AWS SSO profile

#os.environ['AWS_PROFILE'] = 'your_profile'
# Remove any existing credential env vars to force profile usage
#for key in ['AWS_ACCESS_KEY_ID', 'AWS_SECRET_ACCESS_KEY', 'AWS_SESSION_TOKEN']:
#    os.environ.pop(key, None)

os.environ['AWS_REGION'] = 'us-east-1'

print("✅ AWS Profile set. Please restart kernel and run all cells.")

In [ ]:
import boto3
import json
import logging
import time
import os
from pathlib import Path

# Verify AWS credentials and region
session = boto3.Session()
region = session.region_name or "us-east-1"

try:
    sts = session.client("sts")
    identity = sts.get_caller_identity()
    print("✅ AWS Credentials Verified")
    print(f"   Account: {identity['Account']}")
    print(f"   ARN:     {identity['Arn']}")
    print(f"   Region:  {region}")
except Exception as e:
    print(f"❌ AWS Credentials Error: {e}")
    print("   Please configure: aws configure")

---
# Part 2: Deploy Infrastructure

We'll deploy **3 Lambda functions** as insurance underwriting tools:

| Tool | Description | Key Parameter |
|------|-------------|---------------|
| `ApplicationTool` | Creates insurance applications | `coverage_amount`, `applicant_region` |
| `RiskModelTool` | Invokes external risk scoring | `API_classification`, `data_governance_approval` |
| `ApprovalTool` | Approves underwriting decisions | `claim_amount`, `risk_level` |

These are simplified mock implementations for demo purposes.

In [ ]:
import zipfile
import io

lambda_client = boto3.client("lambda", region_name=region)
iam_client = boto3.client("iam", region_name=region)
account_id = boto3.client("sts").get_caller_identity()["Account"]

# ── Helper: create or get Lambda execution role ──────────────────────────
def create_lambda_role(role_name):
    trust_policy = json.dumps({
        "Version": "2012-10-17",
        "Statement": [{
            "Effect": "Allow",
            "Principal": {"Service": "lambda.amazonaws.com"},
            "Action": "sts:AssumeRole"
        }]
    })
    try:
        role = iam_client.create_role(
            RoleName=role_name,
            AssumeRolePolicyDocument=trust_policy,
            Description="Lambda execution role for AgentCore Policy Lab"
        )
        iam_client.attach_role_policy(
            RoleName=role_name,
            PolicyArn="arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole"
        )
        time.sleep(10)  # IAM propagation
        return role["Role"]["Arn"]
    except iam_client.exceptions.EntityAlreadyExistsException:
        return iam_client.get_role(RoleName=role_name)["Role"]["Arn"]

# ── Helper: package and deploy a Lambda ──────────────────────────────────
def deploy_lambda(name, handler_code, role_arn):
    buf = io.BytesIO()
    with zipfile.ZipFile(buf, "w") as zf:
        zf.writestr("index.py", handler_code)
    zip_bytes = buf.getvalue()

    try:
        fn = lambda_client.create_function(
            FunctionName=name,
            Runtime="python3.12",
            Role=role_arn,
            Handler="index.handler",
            Code={"ZipFile": zip_bytes},
            Timeout=30,
            Description=f"AgentCore Policy Lab - {name}"
        )
        print(f"  ✅ Created: {name}")
    except lambda_client.exceptions.ResourceConflictException:
        lambda_client.update_function_code(FunctionName=name, ZipFile=zip_bytes)
        fn = lambda_client.get_function(FunctionName=name)["Configuration"]
        print(f"  ♻️  Updated: {name}")
    return fn["FunctionArn"]

# ── Lambda handler code ───────────────────────────────────────────────────
APPLICATION_TOOL_CODE = '''
import json

def handler(event, context):
    body = json.loads(event.get("body", "{}")) if isinstance(event.get("body"), str) else event
    region = body.get("applicant_region", "UNKNOWN")
    amount = body.get("coverage_amount", 0)
    app_id = f"APP-{region}-{int(amount/1000)}K"
    return {
        "statusCode": 200,
        "body": json.dumps({
            "application_id": app_id,
            "status": "CREATED",
            "message": f"Insurance application created for {region} with ${amount:,.0f} coverage"
        })
    }
'''

RISK_MODEL_TOOL_CODE = '''
import json

def handler(event, context):
    body = json.loads(event.get("body", "{}")) if isinstance(event.get("body"), str) else event
    api_class = body.get("API_classification", "unknown")
    dg_approval = body.get("data_governance_approval", False)
    risk_score = 0.25 if (api_class == "public" and dg_approval) else 0.75
    return {
        "statusCode": 200,
        "body": json.dumps({
            "risk_score": risk_score,
            "risk_level": "LOW" if risk_score < 0.5 else "HIGH",
            "model_version": "v2.1",
            "message": f"Risk model invoked (API class: {api_class}, DG approval: {dg_approval})"
        })
    }
'''

APPROVAL_TOOL_CODE = '''
import json

def handler(event, context):
    body = json.loads(event.get("body", "{}")) if isinstance(event.get("body"), str) else event
    amount = body.get("claim_amount", 0)
    risk = body.get("risk_level", "unknown")
    approved = amount <= 500000 and risk in ["low", "medium"]
    return {
        "statusCode": 200,
        "body": json.dumps({
            "decision": "APPROVED" if approved else "ESCALATED",
            "claim_amount": amount,
            "risk_level": risk,
            "message": f"Claim of ${amount:,.0f} with {risk} risk: {'APPROVED' if approved else 'ESCALATED to senior underwriter'}"
        })
    }
'''

print("🔧 Creating Lambda execution role...")
role_arn = create_lambda_role("AgentCorePolicyLabLambdaRole")
print(f"   Role ARN: {role_arn}")

print("\n📦 Deploying Lambda functions...")
application_arn = deploy_lambda("AgentCore-Policy-ApplicationTool", APPLICATION_TOOL_CODE, role_arn)
risk_model_arn   = deploy_lambda("AgentCore-Policy-RiskModelTool",   RISK_MODEL_TOOL_CODE,   role_arn)
approval_arn     = deploy_lambda("AgentCore-Policy-ApprovalTool",     APPROVAL_TOOL_CODE,     role_arn)

print(f"\n✅ Lambda functions deployed:")
print(f"   ApplicationTool: {application_arn}")
print(f"   RiskModelTool:   {risk_model_arn}")
print(f"   ApprovalTool:    {approval_arn}")

In [ ]:
from bedrock_agentcore_starter_toolkit.operations.gateway.client import GatewayClient
import sys

sys.path.append('../backend')

from cognito_config import activate_oauth_client_credentials

gateway_client = GatewayClient(region_name=region)
gateway_client.logger.setLevel(logging.WARNING)

def get_project_root():
    cwd = Path.cwd().resolve()
    if (cwd / "backend").exists() and cwd.name == "capstone_project":
        return cwd
    if (cwd / "capstone_project" / "backend").exists():
        return cwd / "capstone_project"
    if cwd.name == "notebooks" and (cwd.parent / "backend").exists():
        return cwd.parent
    raise FileNotFoundError("Could not locate the capstone_project backend directory from the current working directory.")

PROJECT_DIR = get_project_root()
POLICY_DIR = PROJECT_DIR / "backend" / "policy"
POLICY_DIR.mkdir(parents=True, exist_ok=True)

GATEWAY_NAME = "InsuranceUnderwritingGateway"
CONFIG_FILE = POLICY_DIR / "policy_lab_config.json"

def get_gateway_summary_by_name(name):
    gateways = gateway_client.client.list_gateways().get("items", [])
    return next((g for g in gateways if g["name"] == name), None)

def get_gateway_authorizer_details(gateway):
    custom_jwt = gateway.get("authorizerConfiguration", {}).get("customJWTAuthorizer", {})
    discovery_url = custom_jwt.get("discoveryUrl")
    allowed_clients = custom_jwt.get("allowedClients", [])
    user_pool_id = None
    if discovery_url and "/" in discovery_url:
        user_pool_id = discovery_url.rstrip("/").split("/")[-2]
    return {
        "discovery_url": discovery_url,
        "allowed_clients": allowed_clients,
        "user_pool_id": user_pool_id,
    }

def gateway_matches_client_info(gateway, candidate_client_info):
    details = get_gateway_authorizer_details(gateway)
    return (
        candidate_client_info.get("client_id") in details["allowed_clients"]
        and candidate_client_info.get("user_pool_id") == details["user_pool_id"]
    )

def delete_user_pool_if_present(user_pool_id):
    if not user_pool_id:
        return
    cognito_client = boto3.client("cognito-idp", region_name=region)
    try:
        cognito_client.delete_user_pool(UserPoolId=user_pool_id)
        print(f"   ✅ Deleted Cognito user pool: {user_pool_id}")
    except cognito_client.exceptions.ResourceNotFoundException:
        pass
    except Exception as delete_error:
        print(f"   ⚠️  Cognito cleanup warning for {user_pool_id}: {str(delete_error)[:120]}")

print(f"🚀 Preparing AgentCore Gateway: {GATEWAY_NAME}")

existing_lab_config = None
gateway = None
client_info = None

if os.path.exists(CONFIG_FILE):
    try:
        with open(CONFIG_FILE) as f:
            existing_lab_config = json.load(f)
        existing_gateway_id = existing_lab_config.get("gateway", {}).get("gateway_id")
        if existing_gateway_id:
            gateway = gateway_client.client.get_gateway(gatewayIdentifier=existing_gateway_id)
            client_info = existing_lab_config["gateway"]["client_info"]
            if gateway_matches_client_info(gateway, client_info):
                activate_oauth_client_credentials(client_info, region=region)
                print("   ✅ Reusing existing lab gateway and Cognito configuration")
            else:
                details = get_gateway_authorizer_details(gateway)
                print("   ⚠️  Existing config does not match the gateway authorizer. Recreating lab resources...")
                gateway_client.cleanup_gateway(existing_gateway_id, client_info)
                delete_user_pool_if_present(details["user_pool_id"])
                if os.path.exists(CONFIG_FILE):
                    os.remove(CONFIG_FILE)
                gateway = None
                client_info = None
    except Exception as reuse_error:
        print(f"   ⚠️  Existing config could not be reused: {str(reuse_error)[:120]}")
        gateway = None
        client_info = None
        existing_summary = get_gateway_summary_by_name(GATEWAY_NAME)
        if existing_summary:
            try:
                stale_gateway = gateway_client.client.get_gateway(gatewayIdentifier=existing_summary["gatewayId"])
                stale_details = get_gateway_authorizer_details(stale_gateway)
                gateway_client.cleanup_gateway(existing_summary["gatewayId"], existing_lab_config.get("gateway", {}).get("client_info") if existing_lab_config else None)
                delete_user_pool_if_present(stale_details["user_pool_id"])
            except Exception as cleanup_error:
                print(f"   ⚠️  Stale gateway cleanup warning: {str(cleanup_error)[:120]}")
        if os.path.exists(CONFIG_FILE):
            os.remove(CONFIG_FILE)

if gateway is None:
    print("🔐 Creating inbound OAuth with Cognito for this lab...")
    cognito_result = gateway_client.create_oauth_authorizer_with_cognito(GATEWAY_NAME)
    client_info = cognito_result["client_info"]
    activate_oauth_client_credentials(client_info, region=region)
    authorizer_config = cognito_result.get("authorizer_config") or cognito_result.get("authorization")

    if not authorizer_config:
        raise ValueError("Missing Cognito authorizer configuration from create_oauth_authorizer_with_cognito()")

    gateway = gateway_client.create_mcp_gateway(
        name=GATEWAY_NAME,
        role_arn=None,
        authorizer_config=authorizer_config
    )
    print("   ✅ Gateway created")

GATEWAY_ID = gateway["gatewayId"]
GATEWAY_ARN = gateway["gatewayArn"]
GATEWAY_URL = gateway["gatewayUrl"]

lab_config = {
    "gateway": {
        "gateway_url": GATEWAY_URL,
        "gateway_id": GATEWAY_ID,
        "region": region,
        "client_info": client_info
    },
    "policy_engine_id": None,
    "policy_engine_arn": None
}

with open(CONFIG_FILE, "w") as f:
    json.dump(lab_config, f, indent=2)

print(f"\n   Gateway ID:  {GATEWAY_ID}")
print(f"   Gateway URL: {GATEWAY_URL}")
print(f"   Config saved to: {CONFIG_FILE}")
print(f"   OAuth client ID: {client_info['client_id']}")
print("   This advanced lab manages its own Cognito setup and can safely reuse its saved config on reruns.")


In [ ]:
# ── Tool schemas for each Lambda target ───────────────────────────────────
def make_tool_schema(tool_name, description, properties, required):
    return {
        "name": tool_name,
        "description": description,
        "inputSchema": {
            "type": "object",
            "properties": properties,
            "required": required
        }
    }

tool_schemas = {
    "ApplicationToolTarget": make_tool_schema(
        "create_application",
        "Creates an insurance application",
        {"applicant_region": {"type": "string", "description": "Geographic region (e.g. US, CA, EU)"},
         "coverage_amount": {"type": "integer", "description": "Requested coverage amount in USD"}},
        ["applicant_region", "coverage_amount"]
    ),
    "RiskModelToolTarget": make_tool_schema(
        "invoke_risk_model",
        "Invokes the risk scoring model",
        {"API_classification":       {"type": "string", "description": "API classification: public, internal, or restricted"},
         "data_governance_approval": {"type": "boolean", "description": "Data governance approval status"}},
        ["API_classification", "data_governance_approval"]
    ),
    "ApprovalToolTarget": make_tool_schema(
        "approve_claim",
        "Approves an insurance underwriting decision",
        {"claim_amount": {"type": "integer", "description": "Insurance claim amount in USD"},
         "risk_level":   {"type": "string", "description": "Risk level: low, medium, high, or critical"}},
        ["claim_amount", "risk_level"]
    ),
}

target_arns = {
    "ApplicationToolTarget": application_arn,
    "RiskModelToolTarget":   risk_model_arn,
    "ApprovalToolTarget":    approval_arn,
}

print("🔗 Attaching Lambda targets to Gateway...")
existing_targets = {
    target["name"]
    for target in gateway_client.client.list_gateway_targets(gatewayIdentifier=gateway["gatewayId"]).get("items", [])
}
target_failures = []
for target_name, lambda_arn in target_arns.items():
    if target_name in existing_targets:
        print(f"   ♻️  Reusing existing target: {target_name}")
        continue
    try:
        gateway_client.create_mcp_gateway_target(
            gateway=gateway,
            name=target_name,
            target_type="lambda",
            target_payload={
                "lambdaArn": lambda_arn,
                "toolSchema": {"inlinePayload": [tool_schemas[target_name]]}
            }
        )
        print(f"   ✅ Added target: {target_name}")
    except Exception as e:
        error_message = str(e)
        print(f"   ⚠️  Target {target_name} failed: {error_message[:120]}")
        target_failures.append((target_name, error_message))

if target_failures:
    raise RuntimeError(f"Target creation failed: {target_failures}")

print("\n✅ Gateway infrastructure ready!")

---
# Part 3: Agent WITHOUT Policies — Unrestricted Access

Let's first run the agent **without any Policy Engine** attached.
All three tools should be accessible with no restrictions.

> **Key concept**: Without a Policy Engine, the Gateway passes all requests through directly.

In [ ]:
from strands import Agent
from strands.models import BedrockModel
from strands.tools.mcp.mcp_client import MCPClient
from mcp.client.streamable_http import streamablehttp_client
from bedrock_agentcore_starter_toolkit.operations.gateway.client import GatewayClient

MODEL_ID = "us.anthropic.claude-haiku-4-5-20251001-v1:0"

def get_access_token(gateway_cfg):
    """Get an OAuth access token for the Gateway."""
    client_info = gateway_cfg.get("client_info", {})
    helper_client = GatewayClient(region_name=gateway_cfg.get("region", region))

    try:
        return helper_client.get_access_token_for_cognito(client_info)
    except Exception as helper_error:
        import requests

        resp = requests.post(
            client_info["token_endpoint"],
            data={
                "grant_type": "client_credentials",
                "client_id": client_info["client_id"],
                "client_secret": client_info["client_secret"],
                "scope": client_info.get("scope", "")
            },
            timeout=30
        )
        if resp.status_code != 200:
            raise RuntimeError(
                f"Failed to get Cognito access token ({resp.status_code}): {resp.text[:200]} | helper error: {helper_error}"
            )
        token_payload = resp.json()
        access_token = token_payload.get("access_token")
        if not access_token:
            raise RuntimeError(f"Token response missing access_token: {token_payload}")
        return access_token

def run_agent_with_tools(prompt, gateway_url, access_token):
    """Run a Strands agent connected to the AgentCore Gateway."""
    def _create_streamable_http_transport(headers=None):
        headers = {**headers} if headers else {}
        headers["Authorization"] = f"Bearer {access_token}"
        return streamablehttp_client(gateway_url, headers=headers)

    with MCPClient(_create_streamable_http_transport) as mcp:
        tools = mcp.list_tools_sync()
        tool_names = [getattr(t, "name", getattr(t, "tool_name", type(t).__name__)) for t in tools]
        print(f"   📋 Available tools: {tool_names}")

        model = BedrockModel(
            model_id=MODEL_ID,
            region_name=region
        )
        agent = Agent(
            model=model,
            tools=tools,
            system_prompt="You are an insurance underwriting assistant. Use the available tools to help with insurance tasks."
        )
        response = agent(prompt)
        return str(response)

if os.path.exists(CONFIG_FILE):
    with open(CONFIG_FILE) as f:
        lab_config = json.load(f)
    access_token = get_access_token(lab_config["gateway"])

    print("🤖 Running agent WITHOUT policies (unrestricted)...\n")
    print("Test 1: List tools")
    result = run_agent_with_tools(
        "What tools do you have available?",
        lab_config["gateway"]["gateway_url"],
        access_token
    )
    print(f"   Response: {result[:300]}")
else:
    print("ℹ️  Run Part 2 first to create the Gateway and OAuth configuration.")


In [ ]:
# Test all three tools — no restrictions yet!
if os.path.exists(CONFIG_FILE):
    access_token = get_access_token(lab_config["gateway"])
    gw_url = lab_config["gateway"]["gateway_url"]

    print("🧪 Testing all tools WITHOUT policy restrictions...\n")

    print("Test 2: Large coverage application ($5M — would be blocked by policy later)")
    r2 = run_agent_with_tools(
        "Create an application for US region with $5 million coverage.",
        gw_url, access_token
    )
    print(f"   Result: {r2[:200]}\n")

    print("Test 3: Risk model invocation")
    r3 = run_agent_with_tools(
        "Invoke the risk model with public API classification and data governance approval set to true.",
        gw_url, access_token
    )
    print(f"   Result: {r3[:200]}\n")

    print("Test 4: Claim approval")
    r4 = run_agent_with_tools(
        "Approve a claim for $75,000 with medium risk level.",
        gw_url, access_token
    )
    print(f"   Result: {r4[:200]}\n")

    print("\n💡 OBSERVATION: Without policies, ALL tools are accessible.")
    print("   The agent can create ANY coverage amount — no limits.")
    print("   This is the problem we're about to solve with Cedar policies.")

---
# Part 4: Create and Attach a Policy Engine

Now let's add a **Policy Engine** to the Gateway.

**Key concept**: Once a Policy Engine is attached:
- **Default Deny** kicks in immediately
- An empty Policy Engine = **ALL tools blocked**
- You must explicitly write `permit` policies to allow tool access

In [ ]:
from bedrock_agentcore_starter_toolkit.operations.policy.client import PolicyClient

agentcore_ctrl = boto3.client("bedrock-agentcore-control", region_name=region)
policy_admin_client = PolicyClient(region_name=region)

print("🔧 Creating Policy Engine...")

engines = agentcore_ctrl.list_policy_engines().get("policyEngines", [])
existing_engine = next((e for e in engines if e["name"] == "InsuranceUnderwritingPolicyEngine"), None)
if existing_engine:
    print("   ♻️  Existing Policy Engine found — cleaning it up for a fresh rerun")
    try:
        existing_gateway = agentcore_ctrl.get_gateway(gatewayIdentifier=GATEWAY_ID)
        detach_request = {
            "gatewayIdentifier": existing_gateway["gatewayId"],
            "name": existing_gateway["name"],
            "roleArn": existing_gateway["roleArn"],
            "protocolType": existing_gateway["protocolType"],
            "authorizerType": existing_gateway["authorizerType"],
        }
        for field in ["description", "authorizerConfiguration", "protocolConfiguration", "kmsKeyArn", "customTransformConfiguration", "interceptorConfigurations", "exceptionLevel"]:
            if field in existing_gateway:
                detach_request[field] = existing_gateway[field]
        agentcore_ctrl.update_gateway(**detach_request)
        print("   ✅ Detached existing Policy Engine from Gateway")
    except Exception as detach_error:
        print(f"   ⚠️  Gateway detach warning: {str(detach_error)[:120]}")
    policy_admin_client.cleanup_policy_engine(existing_engine["policyEngineId"])
    for _ in range(30):
        remaining_engines = agentcore_ctrl.list_policy_engines().get("policyEngines", [])
        if not any(engine["name"] == "InsuranceUnderwritingPolicyEngine" for engine in remaining_engines):
            print("   ✅ Previous Policy Engine fully removed")
            break
        time.sleep(2)
    else:
        raise TimeoutError("Timed out waiting for the previous Policy Engine to be deleted")

engine = agentcore_ctrl.create_policy_engine(
    name="InsuranceUnderwritingPolicyEngine",
    description="Policy engine controlling insurance underwriting agent tool access",
    tags={"Environment": "Lab", "Module": "AgentCorePolicy"}
)
print("   ✅ Policy Engine Created")

POLICY_ENGINE_ID = engine["policyEngineId"]
POLICY_ENGINE_ARN = engine["policyEngineArn"]
print(f"   ID:  {POLICY_ENGINE_ID}")
print(f"   ARN: {POLICY_ENGINE_ARN}")

if os.path.exists(CONFIG_FILE):
    with open(CONFIG_FILE) as f:
        lab_config = json.load(f)
    lab_config["policy_engine_id"] = POLICY_ENGINE_ID
    lab_config["policy_engine_arn"] = POLICY_ENGINE_ARN
    with open(CONFIG_FILE, "w") as f:
        json.dump(lab_config, f, indent=2)


In [ ]:
from bedrock_agentcore_starter_toolkit.operations.gateway.client import GatewayClient

gw_client = GatewayClient(region_name=region)

print("🔗 Attaching Policy Engine to Gateway in ENFORCE mode...")
print("   ⚠️  After this: ALL tools will be BLOCKED (default-deny!)\n")

for _ in range(30):
    current_engine = agentcore_ctrl.get_policy_engine(policyEngineId=POLICY_ENGINE_ID)
    if current_engine.get("status") == "ACTIVE":
        break
    time.sleep(2)
else:
    raise TimeoutError(f"Policy Engine {POLICY_ENGINE_ID} did not become ACTIVE in time")

gw_client.update_gateway_policy_engine(
    gateway_identifier=GATEWAY_ID,
    policy_engine_arn=POLICY_ENGINE_ARN,
    mode="ENFORCE"
)

print("✅ Policy Engine attached in ENFORCE mode")
print("")
print("Now let\'s see what happens when the agent tries to list tools...")

In [ ]:
# Demonstrate DEFAULT DENY — empty Policy Engine blocks everything
import requests

def list_available_tools_raw(gateway_url, access_token):
    """Call the Gateway tool list endpoint directly."""
    resp = requests.post(
        gateway_url,
        headers={
            "Authorization": f"Bearer {access_token}",
            "Content-Type": "application/json"
        },
        json={
            "jsonrpc": "2.0",
            "id": "list-tools",
            "method": "tools/list",
            "params": {}
        },
        timeout=15
    )
    return resp.json()

if os.path.exists(CONFIG_FILE):
    with open(CONFIG_FILE) as f:
        lab_config = json.load(f)
    access_token = get_access_token(lab_config["gateway"])

    print("🔍 Listing available tools (with EMPTY Policy Engine attached)...\n")
    tool_list = list_available_tools_raw(lab_config["gateway"]["gateway_url"], access_token)
    print(f"Raw Gateway Response:")
    print(json.dumps(tool_list, indent=2))

    tools = tool_list.get("result", {}).get("tools", [])
    print(f"\n📋 Number of available tools: {len(tools)}")

    if len(tools) == 0:
        print("\n✅ DEFAULT DENY CONFIRMED!")
        print("   The empty Policy Engine is blocking ALL tool discovery.")
        print("   Agents cannot see OR call any tools.")
        print("   This is the safe-fail posture.")
    else:
        print(f"\n   Tools visible: {[t['name'] for t in tools]}")

---
# Part 5: Write Cedar Policies

Now let's write Cedar policies to selectively allow tool access.

**Our Rules:**
1. Allow `create_application` only if `coverage_amount ≤ $1,000,000`
2. Allow `invoke_risk_model` only if `data_governance_approval == true`
3. Allow `approve_claim` for all users (no conditions)

**Cedar Action name format:** `TargetName__operation_name`

In [ ]:
from bedrock_agentcore_starter_toolkit.operations.policy.client import PolicyClient

policy_client = PolicyClient(region_name=region)

print("📝 Creating Cedar policies...\n")

# ── Policy 1: Application creation — coverage_amount ≤ $1M ────────────────
cedar_p1 = (
    f'permit(principal, '
    f'action == AgentCore::Action::"ApplicationToolTarget___create_application", '
    f'resource == AgentCore::Gateway::"{GATEWAY_ARN}") '
    f'when {{ context.input.coverage_amount <= 1000000 }};'
)

p1 = policy_client.create_or_get_policy(
    policy_engine_id=POLICY_ENGINE_ID,
    name="policy_create_application",
    description="Allow application creation for coverage ≤ $1M",
    definition={"cedar": {"statement": cedar_p1}}
)
print(f"   ✅ Policy 1: Application Tool (coverage ≤ $1M)")
print(f"      Cedar: {cedar_p1[:100]}...")

# ── Policy 2: Risk model — requires data governance approval ──────────────
cedar_p2 = (
    f'permit(principal, '
    f'action == AgentCore::Action::"RiskModelToolTarget___invoke_risk_model", '
    f'resource == AgentCore::Gateway::"{GATEWAY_ARN}") '
    f'when {{ context.input.data_governance_approval == true }};'
)

p2 = policy_client.create_or_get_policy(
    policy_engine_id=POLICY_ENGINE_ID,
    name="policy_risk_model",
    description="Allow risk model only with data governance approval",
    definition={"cedar": {"statement": cedar_p2}}
)
print(f"\n   ✅ Policy 2: Risk Model Tool (data governance required)")

# ── Policy 3: Approval tool — open access ─────────────────────────────────
cedar_p3 = (
    f'permit(principal, '
    f'action == AgentCore::Action::"ApprovalToolTarget___approve_claim", '
    f'resource == AgentCore::Gateway::"{GATEWAY_ARN}");'
)

p3 = policy_client.create_or_get_policy(
    policy_engine_id=POLICY_ENGINE_ID,
    name="policy_approve_claim",
    description="Allow claim approval for all users",
    validation_mode="IGNORE_ALL_FINDINGS",
    definition={"cedar": {"statement": cedar_p3}}
)
print(f"\n   ✅ Policy 3: Approval Tool (open access)")

print("\n🎉 All Cedar policies created!")
print("   Tools should now be discoverable and callable (within policy limits).")

---
# Part 6: Test Policy Enforcement

Now let's test each Cedar rule with explicit allow and deny scenarios.

We expect:
- ✅ $750K coverage → **ALLOWED** (below $1M limit)
- ❌ $1.5M coverage → **DENIED** (exceeds $1M limit)
- ✅ Risk model with approval=true → **ALLOWED**
- ❌ Risk model with approval=false → **DENIED**


In [ ]:
print("=" * 60)
print("TEST 1: ALLOW Scenario — $750K Coverage (≤ $1M limit)")
print("=" * 60)
print("Expected: Cedar evaluates 750000 <= 1000000 → TRUE → ALLOW")
print()

if os.path.exists(CONFIG_FILE):
    with open(CONFIG_FILE) as f:
        lab_config = json.load(f)
    access_token = get_access_token(lab_config["gateway"])

    result = run_agent_with_tools(
        "Create an application for US region with $750,000 coverage.",
        lab_config["gateway"]["gateway_url"],
        access_token
    )
    print(f"Agent Response: {result}")
    print()
    print("✅ If you see a successful application creation above → Policy ALLOWED it!")

In [ ]:
print("=" * 60)
print("TEST 2: DENY Scenario — $1.5M Coverage (> $1M limit)")
print("=" * 60)
print("Expected: Cedar evaluates 1500000 <= 1000000 → FALSE → DENY")
print()

if os.path.exists(CONFIG_FILE):
    access_token = get_access_token(lab_config["gateway"])

    result = run_agent_with_tools(
        "Create an application for US region with $1.5 million coverage.",
        lab_config["gateway"]["gateway_url"],
        access_token
    )
    print(f"Agent Response: {result}")
    print()
    print("❌ The agent should report that the action was denied or not possible.")
    print("   The Lambda was NEVER called — Cedar blocked it at the Gateway.")

In [ ]:
print("=" * 60)
print("TEST 3: ALLOW Scenario — Risk Model With Approval")
print("=" * 60)
print("Expected: Cedar evaluates data_governance_approval == true → ALLOW")
print()

if os.path.exists(CONFIG_FILE):
    with open(CONFIG_FILE) as f:
        lab_config = json.load(f)
    access_token = get_access_token(lab_config["gateway"])

    result = run_agent_with_tools(
        "Invoke the risk model with public API classification and data governance approval set to true.",
        lab_config["gateway"]["gateway_url"],
        access_token
    )
    print(f"Agent Response: {result}")
    print()
    print("✅ If you see a successful risk-model response above → Policy ALLOWED it!")


In [ ]:
print("=" * 60)
print("TEST 4: DENY Scenario — Risk Model Without Approval")
print("=" * 60)
print("Expected: Cedar evaluates data_governance_approval == false → DENY")
print()

if os.path.exists(CONFIG_FILE):
    with open(CONFIG_FILE) as f:
        lab_config = json.load(f)
    access_token = get_access_token(lab_config["gateway"])

    result = run_agent_with_tools(
        "Invoke the risk model with internal API classification and data governance approval set to false.",
        lab_config["gateway"]["gateway_url"],
        access_token
    )
    print(f"Agent Response: {result}")
    print()
    print("❌ The agent should report that the risk model invocation was denied or not possible.")
    print("   The Lambda should not be invoked when Cedar blocks the request.")


---
# Part 7: NL2Cedar — Natural Language Policy Authoring

Now let's generate Cedar policies from **plain English** using the Policy Authoring Service.

This is the feature that lets security teams write policies without learning Cedar syntax.

In [ ]:
print("📝 NL2Cedar: Multi-line input → multiple policies\n")

nl_multi = """Allow all users to invoke the risk model tool when data governance approval is true."""

print(f"Natural Language Input:\n{nl_multi}\n")

result_multi = policy_client.generate_policy(
    policy_engine_id=POLICY_ENGINE_ID,
    name=f"nl_multi_{int(time.time())}",
    resource={"arn": GATEWAY_ARN},
    content={"rawText": nl_multi},
    fetch_assets=True
)

if result_multi.get("generatedPolicies"):
    print(f"Generated {len(result_multi['generatedPolicies'])} policy assets:\n")
    for i, gp in enumerate(result_multi["generatedPolicies"], 1):
        cedar = gp.get("definition", {}).get("cedar", {}).get("statement")
        print(f"Policy {i}:")
        if cedar:
            print(cedar)
        else:
            print(json.dumps(gp, indent=2))
        print()
    print("✅ Multi-line NL2Cedar generation completed.")

---
# Part 8: 🧹 Cleanup

Clean up all resources created in this lab.

**Order of operations (important!):**
1. Detach Policy Engine from Gateway
2. Delete policies from Policy Engine
3. Delete the Policy Engine
4. Delete the Gateway (and targets)
5. Delete Lambda functions and IAM role
6. Delete the lab Cognito user pool and local config file

In [ ]:
# Run this cell to clean up all lab resources

print("🧹 Starting cleanup...\n")

# ── Step 1: Detach Policy Engine from Gateway ─────────────────────────────
try:
    print("Step 1: Detaching Policy Engine from Gateway...")
    gw_client.update_gateway_policy_engine(
        gateway_identifier=GATEWAY_ID,
        policy_engine_arn=None,
        mode=None
    )
    print("   ✅ Policy Engine detached")
except Exception as e:
    print(f"   ⚠️  {str(e)[:80]}")

# ── Step 2 & 3: Delete policies then Policy Engine ────────────────────────
try:
    print("\nStep 2: Cleaning up Policy Engine (all policies)...")
    policy_client.cleanup_policy_engine(POLICY_ENGINE_ID)
    print("   ✅ Policy Engine and all policies deleted")
except Exception as e:
    print(f"   ⚠️  {str(e)[:80]}")

# ── Step 4: Delete Gateway ────────────────────────────────────────────────
if os.path.exists(CONFIG_FILE):
    try:
        print("\nStep 3: Cleaning up Gateway...")
        with open(CONFIG_FILE) as f:
            lab_config = json.load(f)
        gw_client.cleanup_gateway(
            lab_config["gateway"]["gateway_id"],
            lab_config["gateway"]["client_info"]
        )
        print("   ✅ Gateway deleted")
    except Exception as e:
        print(f"   ⚠️  {str(e)[:80]}")

# ── Step 5: Delete Lambda functions ──────────────────────────────────────
print("\nStep 4: Deleting Lambda functions...")
for fn_name in ["AgentCore-Policy-ApplicationTool", "AgentCore-Policy-RiskModelTool", "AgentCore-Policy-ApprovalTool"]:
    try:
        lambda_client.delete_function(FunctionName=fn_name)
        print(f"   ✅ Deleted: {fn_name}")
    except Exception as e:
        print(f"   ⚠️  {fn_name}: {str(e)[:60]}")

# ── Step 6: Delete IAM role ───────────────────────────────────────────────
print("\nStep 5: Cleaning up IAM role...")
try:
    iam_client.detach_role_policy(
        RoleName="AgentCorePolicyLabLambdaRole",
        PolicyArn="arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole"
    )
    iam_client.delete_role(RoleName="AgentCorePolicyLabLambdaRole")
    print("   ✅ IAM role deleted")
except Exception as e:
    print(f"   ⚠️  {str(e)[:80]}")

# ── Step 7: Delete Cognito lab resources and local config ─────────────────
print("\nStep 6: Cleaning up Cognito lab resources...")
if os.path.exists(CONFIG_FILE):
    try:
        with open(CONFIG_FILE) as f:
            lab_config = json.load(f)
        client_info = lab_config["gateway"]["client_info"]
        cognito_client = boto3.client("cognito-idp", region_name=lab_config["gateway"].get("region", region))
        cognito_client.delete_user_pool(UserPoolId=client_info["user_pool_id"])
        print(f"   ✅ Deleted Cognito user pool: {client_info['user_pool_id']}")
    except Exception as e:
        print(f"   ⚠️  Cognito cleanup: {str(e)[:80]}")

    try:
        os.remove(CONFIG_FILE)
        print(f"   ✅ Removed local config: {CONFIG_FILE}")
    except Exception as e:
        print(f"   ⚠️  Local config cleanup: {str(e)[:80]}")

print("\n🎉 Cleanup complete!")

---
# 🎉 Lab Complete!

## What You Accomplished

| Step | Concept | Result |
|------|---------|--------|
| Part 2 | Gateway + Lambda targets | Infrastructure deployed |
| Part 3 | No Policy Engine | All tools accessible (no limits) |
| Part 4 | Policy Engine attached | Default Deny — all tools blocked |
| Part 5 | Cedar policies created | Specific tools selectively unlocked |
| Part 6 | ALLOW test ($750K) | Policy permits — application created |
| Part 6 | DENY test ($1.5M) | Policy blocks — Lambda never called |
| Part 7 | NL2Cedar | Natural language → Cedar policies |
| Part 8 | Emergency shutdown | One forbid line blocks everything |

## Key Takeaways

- 🛡️ **Cedar policies operate outside the LLM** — prompt injection cannot bypass them
- 🚫 **Default Deny**: empty Policy Engine = all tools blocked
- ⚡ **Emergency shutdown** = one line of Cedar: `forbid(principal, action, resource);`
- 🗣️ **NL2Cedar** converts plain English to valid Cedar — no syntax expertise needed
- 🎯 **Test first with LOG_ONLY** before switching to ENFORCE in production
